# Amazon Bedrock AgentCore Runtime에 MCP Server 호스팅 - AWS IAM Inbound Authentication

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime에 MCP(Model Context Protocol) server를 호스팅하는 방법을 알아봅니다. Amazon Bedrock AgentCore Python SDK를 사용하여 MCP tool을 Amazon Bedrock AgentCore와 호환되는 MCP server로 래핑합니다.

Amazon Bedrock AgentCore Python SDK가 MCP server 구현 세부 사항을 처리하므로 tool의 핵심 기능에 집중할 수 있습니다. 이 SDK는 직접 통신할 수 있도록 코드를 AgentCore의 표준화된 MCP protocol contract로 변환합니다.

기존 [MCP protocol](https://modelcontextprotocol.io/docs/getting-started/intro) specification에서는 인증에 OAuth token이 필요하지만, AgentCore Runtime은 MCP server로 들어오는 request에 AWS IAM credentials를 구성할 수 있어 중요한 enterprise 요구 사항을 충족합니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Tool 호스팅                                               |
| Tool 유형           | MCP server                                                |
| 튜토리얼 구성 요소  | AgentCore Runtime에 MCP server 호스팅                    |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 쉬움                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 MCP                |

### 튜토리얼 아키텍처

이 튜토리얼에서는 MCP server를 AgentCore Runtime에 배포하는 방법을 설명합니다.

시연을 위해 `add_numbers`, `multiply_numbers`, `greet_user`의 세 가지 tool이 포함된 간단한 MCP server를 사용합니다.

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### 튜토리얼 주요 기능

* custom tool로 MCP server 생성
* 로컬에서 MCP server 테스트
* Amazon Bedrock AgentCore Runtime에 MCP server 호스팅
* 인증을 사용하여 배포된 MCP server 호출


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials 구성 완료
* Amazon Bedrock AgentCore SDK
* MCP(Model Context Protocol) library
* 실행 중인 Docker daemon

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client("ssm", region_name=region)

tool_name = "mcp_server_iam"

## MCP(Model Context Protocol) 이해

MCP는 AI 모델이 외부 데이터와 tool에 안전하게 액세스하도록 지원하는 protocol입니다. 주요 개념은 다음과 같습니다.

* **Tools**: AI가 작업을 수행하기 위해 호출할 수 있는 함수
* **Streamable HTTP**: AgentCore Runtime에서 사용하는 transport protocol
* **Session Isolation**: 각 client가 `Mcp-Session-Id` header를 통해 격리된 session을 사용
* **Stateless Operation**: 확장성을 위해 server가 stateless operation을 지원해야 함

AgentCore Runtime은 MCP server가 기본 path인 `0.0.0.0:8000/mcp`에 호스팅되기를 기대합니다.

### 프로젝트 구조

프로젝트를 다음과 같은 구조로 설정합니다.

```
mcp_server_project/
├── mcp_server.py              # 주요 MCP server 코드
├── mcp_client.py          # 로컬 test client
├── mcp_client_remote.py   # 원격 test client
├── requirements.txt          # 의존성
└── __init__.py              # Python package 표시 파일
```

## MCP Server 생성

간단한 tool 세 개가 포함된 MCP server를 생성합니다. 이 server는 AgentCore Runtime 호환성에 필요한 `stateless_http=True` 설정으로 FastMCP를 사용합니다.

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """Greet a user by name"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### 코드 동작 설명

* **FastMCP**: tool을 호스팅할 수 있는 MCP server 생성
* **@mcp.tool()**: Python 함수를 MCP tool로 변환하는 decorator
* **stateless_http=True**: AgentCore Runtime 호환성에 필요
* **Tools**: 서로 다른 유형의 작업을 보여주는 간단한 tool 세 개

## 로컬 테스트 Client 생성

AgentCore Runtime에 배포하기 전에 MCP server를 로컬에서 테스트할 client를 생성합니다.

In [ ]:
%%writefile mcp_client.py
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

 ### 로컬 테스트

MCP server를 로컬에서 테스트하려면 다음을 수행합니다.

1. **Terminal 1**: MCP server 시작
   ```bash
   python mcp_server.py
   ```
   
2. **Terminal 2**: test client 실행
   ```bash
   python mcp_client.py
   ```

output에 세 가지 tool 목록이 표시됩니다.

## AgentCore Runtime 배포 구성

다음으로 starter toolkit을 사용하여 entrypoint, 방금 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 또한 시작할 때 Amazon ECR repository를 자동으로 생성하도록 starter toolkit을 구성합니다.

configure 단계에서 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
import os
from bedrock_agentcore_starter_toolkit import Runtime

print(f"Using AWS region: {region}")

required_files = ["mcp_server.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed ✓")

## AgentCore Runtime에 MCP Server 시작

Dockerfile이 준비되었으므로 MCP server를 AgentCore Runtime에 시작합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

In [ ]:
agent_arn_response = ssm_client.put_parameter(
    Name="/mcp_server/runtime_iam/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for MCP server with inbound auth",
    Overwrite=True,
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

### Session Lifecycle 시연: Session 중지

Runtime이 배포되었으므로 session 중지를 시연합니다. custom session ID로
MCP server를 호출한 다음 microVM 리소스를 해제하도록 session을 중지하되,
새 session에 사용할 수 있도록 Runtime은 계속 실행합니다.

In [ ]:
import uuid
from streamable_http_sigv4 import streamablehttp_client_with_sigv4
from mcp import ClientSession

# custom session ID 생성
demo1_session_id = str(uuid.uuid4())
print(f"📝 Demo 1 - Generated mcpSessionId: {demo1_session_id}")

# MCP URL 준비
encoded_arn = launch_result.agent_arn.replace(":", "%3A").replace("/", "%2F")
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

credentials = boto_session.get_credentials()
headers = {"Mcp-Session-Id": demo1_session_id}


# custom session ID로 호출
async def test_session():
    async with streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service="bedrock-agentcore",
        region=region,
        headers=headers,
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"✅ Session active with {len(tools.tools)} tools")


await test_session()

# session 중지
print(f"\n🛑 Stopping session '{demo1_session_id}'...")
agentcore_client = boto_session.client("bedrock-agentcore", region_name=region)
response = agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=demo1_session_id,
    qualifier="DEFAULT",
)
print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
print("💡 Runtime remains alive for new sessions")

# 참고: 위 log에 'Session termination failed: 404'가 표시될 수 있음
# 이는 예상된 동작으로, 이미 session을 중지한 후 MCP client가 자동 정리를 시도하기 때문임
# 중요한 부분은 명시적 stop_runtime_session 호출의 'HTTP 200' 응답임

## 원격 테스트 Client 생성

배포된 MCP server를 테스트할 client를 생성합니다. 이 client는 AWS에서 필요한 credentials를 가져와 배포된 server에 연결합니다.

In [ ]:
%%writefile mcp_client_remote.py       
import asyncio
import sys
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4


logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    AWS SigV4 인증을 사용하는 streamable HTTP transport를 생성합니다.

    이 함수는 AWS Signature Version 4(SigV4)로 요청을 인증하는 MCP client transport를
    생성합니다. 표준 MCP 클라이언트는 AWS IAM 인증을 기본 지원하지 않으므로,
    이 transport가 그 간극을 연결합니다.

    매개변수:
        mcp_url (str): MCP gateway endpoint URL
        service_name (str): SigV4 서명에 사용할 AWS 서비스 이름(일반적으로 "bedrock-agentcore")
        region (str): gateway가 배포된 AWS 리전

    반환값:
        StreamableHTTPTransportWithSigV4: SigV4 인증용으로 구성된 transport 인스턴스

    예시:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url=".../mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # 현재 boto3 session에서 AWS credentials 가져오기
    # 이 credentials는 SigV4로 request를 signing하는 데 사용됨
    session = boto3.Session()
    credentials = session.get_credentials()

    # SigV4 signing 기능이 있는 custom transport를 생성하여 반환
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    페이지네이션을 처리해 MCP 클라이언트의 전체 도구 목록을 조회합니다.

    MCP 서버는 도구를 페이지 단위 응답으로 반환할 수 있습니다. 이 함수는 페이지네이션을
    자동으로 처리하고 사용 가능한 모든 도구를 하나의 목록으로 반환합니다.

    매개변수:
        client: MCP 클라이언트 인스턴스(strands.tools.mcp.mcp_client.MCPClient)

    반환값:
        list: MCP 서버에서 사용할 수 있는 모든 도구의 전체 목록

    예시:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # 모든 page를 가져올 때까지 반복
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # 가져올 page가 더 있는지 확인
        if tmp_tools.pagination_token is None:
            # page가 더 없으므로 완료
            more_tools = False
        else:
            # page가 더 있으므로 다음 page를 가져올 준비
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    try:
        async with create_streamable_http_transport_sigv4(
            mcp_url=mcp_url, service_name="bedrock-agentcore", region=region
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, "inputSchema") and tool.inputSchema:
                        properties = tool.inputSchema.get("properties", {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()

                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## 배포된 MCP Server 테스트

remote client를 사용하여 배포된 MCP server를 테스트합니다.

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote.py

### MCP Tool 원격 호출

tool 목록을 표시할 뿐 아니라 직접 호출하여 전체 MCP 기능을 보여주는 향상된 client를 생성합니다.

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import sys
import os
import logging
import boto3
import uuid
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """AWS SigV4 인증을 사용하는 streamable HTTP transport를 생성합니다."""
    session = boto3.Session()
    credentials = session.get_credentials()
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    # custom session ID 생성
    mcp_session_id = str(uuid.uuid4())
    print(f"\n📝 Generated custom mcpSessionId: {mcp_session_id}")
    
    # SigV4용 credentials 가져오기
    credentials = boto_session.get_credentials()
    
    # custom session ID를 header로 전달
    headers = {"Mcp-Session-Id": mcp_session_id}

    try:
        async with streamablehttp_client_with_sigv4(
                url=mcp_url,
                credentials=credentials,
                service="bedrock-agentcore",
                region=region,
                headers=headers
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")

                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)

                try:
                    print("\n➕ Testing add_numbers(5, 3)...")
                    add_result = await session.call_tool(
                        name="add_numbers", arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n✖️  Testing multiply_numbers(4, 7)...")
                    multiply_result = await session.call_tool(
                        name="multiply_numbers", arguments={"a": 4, "b": 7}
                    )
                    print(f"   Result: {multiply_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n👋 Testing greet_user('Alice')...")
                    greet_result = await session.call_tool(
                        name="greet_user", arguments={"name": "Alice"}
                    )
                    print(f"   Result: {greet_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                print("\n✅ MCP tool testing completed!")
        
        # session 중지 시연
        print(f"\n🛑 Stopping session '{mcp_session_id}'...")
        agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
        response = agentcore_client.stop_runtime_session(
            agentRuntimeArn=agent_arn,
            runtimeSessionId=mcp_session_id,
            qualifier='DEFAULT'
        )
        print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
        print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
        print(f"   MicroVM resources released")
        print(f"💡 Runtime remains alive for new sessions")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## Tool 호출 테스트

MCP tool을 직접 호출하여 테스트합니다.

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

### Session Lifecycle 시연: 이전 Session 중지

Boto3 방식으로 넘어가기 전에 이전 MCP client 테스트의 session을 중지합니다.
workflow의 어느 시점에서든 session을 중지할 수 있음을 보여줍니다.

In [ ]:
# 이전 mcp_client_remote.py 테스트에서 session을 생성했음
# 서로 다른 테스트 방식 사이의 session 관리를 보여주기 위해 여기서 중지할 수 있음
print("💡 Note: The previous MCP client test created a session that we could stop here.")
print("   In production, track session IDs from your invocations and stop them when done.")
print("   For this demo, we'll create and stop a new session to show the pattern.")

# 시연을 위해 session을 생성한 후 즉시 중지
demo2_session_id = str(uuid.uuid4())
print(f"\n📝 Demo 2 - Generated mcpSessionId: {demo2_session_id}")


async def quick_session():
    async with streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service="bedrock-agentcore",
        region=region,
        headers={"Mcp-Session-Id": demo2_session_id},
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print(f"✅ Session {demo2_session_id} created")


await quick_session()

# 중지
print(f"🛑 Stopping session '{demo2_session_id}'...")
response = agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=demo2_session_id,
    qualifier="DEFAULT",
)
print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
print("   Ready for Boto3 approach test")

## Remote MCP Server 테스트 - Boto3 방식 

이 섹션에서는 Boto3 SDK의 `invoke_agent_runtime` API를 사용하여 배포된 MCP server를 테스트하는 다른 방식을 보여줍니다. SDK가 AWS SigV4 request signing을 자동으로 처리하므로 Runtime 호출을 위한 IAM 인증이 간소화됩니다.

### Remote Testing Client 생성 - Boto3

Boto3 API를 사용하여 배포된 MCP server를 테스트할 client를 생성합니다. 

In [ ]:
%%writefile mcp_client_remote_boto3.py   

import boto3
import json
import traceback
from boto3.session import Session
from botocore.exceptions import ClientError

boto_session = Session()
region = boto_session.region_name
print(f"Using AWS region: {region}")

# Bedrock AgentCore 및 SSM client 초기화
client = boto3.client('bedrock-agentcore', region_name=region)
ssm_client = boto3.client("ssm", region_name=region)


agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
)

runtime_arn = agent_arn_response["Parameter"]["Value"]

print(f"Retrieved Agent ARN: {runtime_arn}")

if not runtime_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)
        
def call_mcp(method, params=None):
    """
    agent runtime에서 MCP 메서드를 호출합니다.
    
    매개변수:
        method: 호출할 MCP 메서드(예: 'tools/list', 'tools/call')
        params: 메서드에 전달할 선택적 매개변수
    
    반환값:
        MCP 응답의 결과
    """
    if params is None:
        params = {}

    payload = json.dumps({
        "jsonrpc": "2.0",
        "id": 1,
        "method": method,
        "params": params
    }).encode()

    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            payload=payload,
            qualifier='DEFAULT',
            contentType='application/json',
            accept='application/json, text/event-stream'
        )

        raw = response['response'].read().decode()
        json_data = json.loads(raw[raw.find('{'):])
        return json_data['result']

    except ClientError as e:
        print(f"\n{'=' * 60}")
        print("Error Response:")
        print(json.dumps(e.response, indent=2, default=str))
        print(f"{'=' * 60}\n")
        raise


def main():

    try:
        # 사용 가능한 tool 목록 표시
        print("📋 Available MCP Tools:")
        print("=" * 50)
        
        tools_result = call_mcp("tools/list")
        tools = tools_result['tools']
        
        for tool in tools:
            params = list(tool.get('inputSchema', {}).get('properties', {}).keys())
            print(f"🔧 {tool['name']}")
            print(f"   Description: {tool['description']}")
            print(f"   Parameters: {params}")
            print()
        
        print(f"✅ Successfully connected to MCP server!")
        print(f"Found {len(tools)} tools available.")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()


### 배포된 MCP Server 테스트

remote client를 사용하여 배포된 MCP server를 테스트합니다.

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote_boto3.py

### MCP Tool 호출 - Boto3

이제 boto3 SDK를 사용하여 tool 목록을 표시할 뿐 아니라 직접 호출하는 client를 생성합니다.

In [ ]:
%%writefile invoke_mcp_tools_boto3.py

import boto3
import json
import logging
from boto3.session import Session
from botocore.exceptions import ClientError

boto_session = Session()
region = boto_session.region_name
client = boto3.client('bedrock-agentcore', region_name=region)

ssm_client = boto3.client("ssm", region_name=region)
agent_arn_response = ssm_client.get_parameter(Name="/mcp_server/runtime_iam/agent_arn")
runtime_arn = agent_arn_response["Parameter"]["Value"]

def call_mcp(method, params=None):
    if params is None:
        params = {}
    payload = json.dumps({
        "jsonrpc": "2.0",
        "id": 1,
        "method": method,
        "params": params
    }).encode()
    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=runtime_arn,
            payload=payload,
            qualifier='DEFAULT',
            contentType='application/json',
            accept='application/json, text/event-stream'
        )
        raw = response['response'].read().decode()
        json_data = json.loads(raw[raw.find('{'):])
        return json_data['result']
    except ClientError as e:
        print(f"❌ Error: {e}")
        raise

def main():
    
    print(f"Using AWS region: {region}")
    print(f"Retrieved Agent ARN: {runtime_arn}")


    print("\n🔄 Listing available tools...")
    try: 
        tools_result = call_mcp("tools/list")

        print("\n📋 Available MCP Tools:")
        print("=" * 50)
        for tool in tools_result['tools']:
            print(f"🔧 {tool['name']}: {tool['description']}")

        print("\n🧪 Testing MCP Tools:")
        print("=" * 50)

        print("\n➕ Testing add_numbers(5, 3)...")
        add_result = call_mcp("tools/call", {"name": "add_numbers", "arguments": {"a": 5, "b": 3}})
        print(f"   Result: {add_result['structuredContent']}")

        print("\n✖️  Testing multiply_numbers(4, 7)...")
        multiply_result = call_mcp("tools/call", {"name": "multiply_numbers", "arguments": {"a": 4, "b": 7}})
        print(f"   Result: {multiply_result['structuredContent']}")

        print("\n👋 Testing greet_user('Alice')...")
        greet_result = call_mcp("tools/call", {"name": "greet_user", "arguments": {"name": "Alice"}})
        print(f"   Result: {greet_result['structuredContent']}")

        print("\n✅ MCP tool testing completed!")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)

if __name__ == "__main__":
    main()


### Tool 호출 테스트

새로 생성한 client를 호출하여 MCP tool을 테스트합니다.

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools_boto3.py

### Session Lifecycle 시연: Boto3 테스트 후 중지

Boto3 방식으로 테스트한 후 다른 session을 중지하는 방법을 시연합니다.
서로 다른 호출 방식에서도 session 관리가 일관되게 작동함을 보여줍니다.

In [ ]:
demo3_session_id = str(uuid.uuid4())
print(f"📝 Demo 3 - Generated mcpSessionId: {demo3_session_id}")


async def test3():
    async with streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service="bedrock-agentcore",
        region=region,
        headers={"Mcp-Session-Id": demo3_session_id},
    ) as (r, w, _):
        async with ClientSession(r, w) as s:
            await s.initialize()
            print("✅ Session created")


await test3()

print(f"🛑 Stopping session '{demo3_session_id}'...")
response = agentcore_client.stop_runtime_session(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=demo3_session_id,
    qualifier="DEFAULT",
)
print(f"✅ Session stopped (HTTP {response['ResponseMetadata']['HTTPStatusCode']})")
print(f"   Request ID: {response['ResponseMetadata']['RequestId']}")
print("💡 All demos complete - runtime handled multiple sessions!")

## 다음 단계

MCP server를 AgentCore Runtime에 성공적으로 배포했으므로 다음 작업을 수행할 수 있습니다.

1. **Tool 추가**: MCP server에 tool 추가
2. **Custom 인증**: AWS IAM inbound authentication 구현
3. **통합**: 다른 AgentCore service와 통합

## Session Lifecycle 모범 사례

AgentCore Runtime 비용은 vCPU와 Memory를 기준으로 책정됩니다. 원치 않는 비용을 방지하려면 session을 명시적으로 중지하거나 적절한 idle timeout을 구성하여 session이 종료되도록 하는 것이 좋습니다.

비용을 효과적으로 관리하려면 다음을 수행합니다.

- **idle timeout 구성**: session을 생성할 때 적절한 idle timeout을 설정하여 비활성 session을 자동으로 중지합니다. 사용 사례에 맞는 값(예: 개발/테스트에는 짧게, production workload에는 길게)을 선택합니다.
- **완료 후 session 중지**: Runtime은 새 session에 사용할 수 있도록 유지하면서 `stop_runtime_session`으로 특정 session의 microVM 리소스를 해제합니다.

## 리소스 정리

이제 AgentCore Runtime과 관련 리소스를 정리합니다. 원치 않는 비용을 방지하도록 Runtime을 먼저 삭제한 다음 ECR repository 같은 지원 리소스를 정리합니다.

In [ ]:
# --- 리소스 정리 ---
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# 1단계: credential 노출 시간을 최소화하도록 Parameter Store parameter를 먼저 삭제
try:
    ssm_client.delete_parameter(Name="/mcp_server/runtime_iam/agent_arn")
    print("✅ Parameter Store parameter deleted")
except ssm_client.exceptions.ParameterNotFound:
    print("ℹ️  Parameter Store parameter not found")

# 2단계: 비용 발생을 중지하도록 agent runtime 삭제
# AgentCore Runtime 비용은 vCPU와 Memory를 기준으로 책정됨
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Agent runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete agent runtime: {e}")

# 3단계: ECR repository 삭제
try:
    ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)
    print("✅ ECR repository deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

print("\n✅ Cleanup completed successfully!")

# 🎉 축하합니다!

다음 작업을 성공적으로 완료했습니다.

✅ custom tool로 **MCP server 생성**  
✅ MCP client로 **로컬 테스트**  
✅ Amazon Cognito로 **인증 설정**  
✅ AgentCore Runtime을 사용하여 **AWS에 배포**  
✅ 적절한 인증으로 **원격 호출**  
✅ **MCP 개념과 best practice 학습**  

이제 MCP server가 Amazon Bedrock AgentCore Runtime에서 실행되고 있으며 production에서 사용할 준비가 되었습니다.

## 요약

이 튜토리얼에서는 다음 방법을 학습했습니다.
- FastMCP를 사용하여 MCP server 구축
- AgentCore 호환성을 위한 stateless HTTP transport 구성
- AWS IAM inbound authentication 설정
- AWS에 MCP server를 배포하고 관리
- 로컬 및 원격 테스트
- tool 호출에 MCP client 사용

배포된 MCP server를 이제 더 큰 AI 애플리케이션과 workflow에 통합할 수 있습니다.